# 11: Catboost Feature Engineering 

Builds the game-level training table for the CatBoost attendance model:
table_rank, opponent_table_rank, days_since_last_home_game, day_of_week, venue_name,
home_team, away_team, stadium_capacity, metro_size -> log(attendance).

Restricted to NWSL games (league == 'nwsl'), COVID seasons (2020-2021) excluded (same
convention as 10_attendance_model.ipynb), and to games with a non-null attendance.

Spatial catchment source: the 14 current-NWSL-venue isochrone/tract-travel-time.
Denver Summit FC and Boston Legacy FC (2026 expansion, no
confirmed venue) are dropped because of lack of a complete season.

metro_size = reachable population within 60 min of the venue (catchment_pop_60min)


In [1]:
import os

import numpy as np
import pandas as pd

DATA_PROCESSED_DIR = os.path.join("..", "data", "processed")
CENSUS_DIR = os.path.join("..", "data", "raw", "census")
OUTPUT_DIR = os.path.join("..", "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

COVID_SEASONS = {2020, 2021}

# Venues
TRAVEL_TIME_FILES = [
    "kansas_city_tract_travel_times.csv",
    "san_diego_tract_travel_times.csv",
    "seattle_tacoma_tract_travel_times.csv",
    "dc_tract_travel_times.csv",
    "nyc_tract_travel_times_90min.csv",
    "san_jose_tract_travel_times.csv",
    "los_angeles_tract_travel_times.csv",
    "houston_tract_travel_times.csv",
    "chicago_tract_travel_times.csv",
    "portland_tract_travel_times.csv",
    "orlando_tract_travel_times.csv",
    "salt_lake_tract_travel_times.csv",
    "louisville_tract_travel_times.csv",
    "raleigh_cary_tract_travel_times.csv",
]

ACS_FILES = [
    "acs_population_by_tract_comparison_cities.csv",  # MO, KS, CA, WA, MD, DC
    "acs_population_by_tract_new_venues.csv",  # TX, IL, OR, FL, UT, KY, NC
]
ACS_NY_NJ = os.path.join(CENSUS_DIR, "acs_population_by_tract.csv")

In [2]:
print("=== 1. Catchment population (30/60 min) per venue ===")
acs_frames = [pd.read_csv(os.path.join(CENSUS_DIR, f), dtype={"GEOID": str}) for f in ACS_FILES]
if os.path.exists(ACS_NY_NJ):
    ny_nj = pd.read_csv(ACS_NY_NJ, dtype=str)
    # this file has no GEOID column (state/county/tract instead) -> construct it
    ny_nj["GEOID"] = (
        ny_nj["state"].str.zfill(2) + ny_nj["county"].str.zfill(3) + ny_nj["tract"].str.zfill(6)
    )
    acs_frames.append(ny_nj)
else:
    print(f"  WARNING: {ACS_NY_NJ} not found -- Gotham catchment population will be incomplete.")
acs = pd.concat(acs_frames, ignore_index=True)[["GEOID", "population"]].drop_duplicates("GEOID")
acs["population"] = pd.to_numeric(acs["population"], errors="coerce")
print(f"  {len(acs)} tracts with population loaded")

tt_frames = []
for fname in TRAVEL_TIME_FILES:
    path = os.path.join(DATA_PROCESSED_DIR, fname)
    if not os.path.exists(path):
        print(f"  WARNING: missing {path}, skipping")
        continue
    df = pd.read_csv(path, dtype={"GEOID": str, "from_id": str})
    tt_frames.append(df[["from_id", "GEOID", "travel_time_minutes"]])
travel_times = pd.concat(tt_frames, ignore_index=True)
travel_times = travel_times.merge(acs, on="GEOID", how="left")

catchment = (
    travel_times.groupby("from_id")
    .apply(lambda g: pd.Series({
        "catchment_pop_30min": g.loc[g["travel_time_minutes"] <= 30, "population"].sum(),
        "catchment_pop_60min": g.loc[g["travel_time_minutes"] <= 60, "population"].sum(),
        "median_travel_time_reachable": g.loc[g["travel_time_minutes"].notna(), "travel_time_minutes"].median(),
    }), include_groups=False)
    .reset_index()
    .rename(columns={"from_id": "stadium_id"})
)
print(catchment)

# Saved in full 
catchment.to_csv(os.path.join(OUTPUT_DIR, "venue_catchment_population.csv"), index=False)

=== 1. Catchment population (30/60 min) per venue ===
  43685 tracts with population loaded
     stadium_id  catchment_pop_30min  catchment_pop_60min  \
0    0Oq62v7q6D              26886.0             358541.0   
1    0x5g6ojM7O              57833.0             549316.0   
2    7vQ7xbOMD1             218728.0            1464640.0   
3    9Yqda07QvJ             111635.0             833559.0   
4    BLMvra8Mxe              26979.0             188743.0   
5   ETIHAD_PARK             448545.0            3201230.0   
6    KXMe8lXQ64              75360.0             782732.0   
7    NPqxy6XQ9d               4307.0               6028.0   
8    NWMW84L5lz              64815.0            1051425.0   
9    Oa5wKXY514               9590.0             402583.0   
10   Oa5wdz9q14               6697.0              77359.0   
11   Vj58W84M8n              15341.0             471028.0   
12   e7MzlRjqr0              12748.0             316627.0   
13   gpMOrLOQzy              11054.0              3968

In [3]:
print("=== 2. Venue supply-side features ===")
stadium_acc = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "stadium_accessibility.csv"))
teams_lookup = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "nwsl_transportation_teams_clean.csv")).set_index("team_id")["league"].to_dict()
venue_cols = stadium_acc[[
    "stadium_id", "stadium_name", "year_built", "capacity",
    "primary_team_id", "secondary_team_id",
]].copy()
# shared_with_other_league: fixed because originally computed wrong
venue_cols["_primary_league"] = venue_cols["primary_team_id"].map(teams_lookup)
venue_cols["_secondary_league"] = venue_cols["secondary_team_id"].map(teams_lookup)
venue_cols["shared_with_other_league"] = venue_cols.apply(
    lambda r: int({"nwsl", "mls"}.issubset({r["_primary_league"], r["_secondary_league"]} - {None})), axis=1
)
venue_cols["capacity"] = pd.to_numeric(venue_cols["capacity"], errors="coerce")
# CPKC Stadium (Kansas City Current) has no capacity, filled it in
venue_cols.loc[venue_cols["stadium_id"] == "xW5p3L0Mg1", "capacity"] = 11500

# Dedicated soccer facility flag: inferred from NFL stadums
SHARED_MULTIPURPOSE = {
    "Lumen Field",          # Seattle Reign, shared with NFL Seahawks
    "Gillette Stadium",     # Boston Legacy, shared with NFL Patriots 
}
venue_cols["dedicated_soccer_facility"] = (~venue_cols["stadium_name"].isin(SHARED_MULTIPURPOSE)).astype(int)

venue_features = venue_cols.merge(catchment, on="stadium_id", how="left")
venue_features["stadium_age_years"] = 2026 - venue_features["year_built"]
venue_features["metro_size"] = venue_features["catchment_pop_60min"]
print(venue_features[["stadium_name", "capacity", "stadium_age_years", "dedicated_soccer_facility",
                       "catchment_pop_30min", "catchment_pop_60min"]])

=== 2. Venue supply-side features ===
                                   stadium_name  capacity  stadium_age_years  \
0                                Torero Stadium       NaN               65.0   
1                          Shell Energy Stadium   22039.0               14.0   
2                              Field of Legends       NaN                NaN   
3                               Lower.com Field   20371.0                5.0   
4                       Rochester Rinos Stadium       NaN                NaN   
5                                   BMO Stadium   22000.0                8.0   
6                                   Lumen Field   38500.0               24.0   
7                              SeatGeek Stadium   20000.0               20.0   
8                                  Jordan Field       NaN                NaN   
9                    Dick's Sporting Goods Park   18081.0               19.0   
10                          Lynn Family Stadium       NaN                6.0   
11

In [4]:
print("=== 3. Game-level table: NWSL games, current 14 venues, non-COVID, non-null attendance ===")
games = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "games_with_transit.csv"), parse_dates=["date_time_utc"])
teams = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "nwsl_transportation_teams_clean.csv"))
team_name = teams.set_index("team_id")["team_name"].to_dict()

# Current 14 teams minus this year's expansion teams
CURRENT_TEAM_NAMES = {
    "Bay FC", "Houston Dash", "Kansas City Current", "San Diego Wave FC", "Seattle Reign FC",
    "Chicago Stars FC", "Portland Thorns FC", "Orlando Pride", "Washington Spirit",
    "Utah Royals FC", "Racing Louisville FC", "Angel City FC", "NJ/NY Gotham FC",
    "North Carolina Courage",
}
current_team_ids = {tid for tid, name in team_name.items() if name in CURRENT_TEAM_NAMES}

games = games[games["league"] == "nwsl"].copy()
games = games[~games["year"].isin(COVID_SEASONS)]
games = games[games["attendance"].notna() & (games["attendance"] > 0)]
games = games[games["home_team_id"].isin(current_team_ids)]
games["home_team"] = games["home_team_id"].map(team_name)
games["away_team"] = games["away_team_id"].map(team_name)
games["day_of_week"] = games["date_time_utc"].dt.day_name()
games["log_attendance"] = np.log(games["attendance"])
print(f"  {len(games)} games after filtering (from {pd.read_csv(os.path.join(DATA_PROCESSED_DIR, 'games_with_transit.csv')).shape[0]} total)")

=== 3. Game-level table: NWSL games, current 14 venues, non-COVID, non-null attendance ===
  1182 games after filtering (from 6981 total)


In [5]:
print("=== 4. Running table_rank / opponent_table_rank (as-of-matchday, no leakage) ===")
def running_standings(games):
    games = games.sort_values("date_time_utc").copy()
    points = {}  # (season, team_id) -> cumulative points BEFORE this game
    games_played = {}
    rank_col, opp_rank_col = [], []
    for _, row in games.iterrows():
        season = row["year"]
        home_pts, away_pts = points.get((season, row["home_team_id"]), 0), points.get((season, row["away_team_id"]), 0)
        # rank = 1 + count of teams (seen so far this season) with strictly more points
        season_pts = {t: p for (s, t), p in points.items() if s == season}
        home_rank = 1 + sum(1 for p in season_pts.values() if p > home_pts)
        away_rank = 1 + sum(1 for p in season_pts.values() if p > away_pts)
        rank_col.append(home_rank)
        opp_rank_col.append(away_rank)
        # update AFTER computing this game's ranks (no leakage from this game's own result)
        if row["home_score"] > row["away_score"]:
            hp, ap = 3, 0
        elif row["home_score"] < row["away_score"]:
            hp, ap = 0, 3
        else:
            hp, ap = 1, 1
        points[(season, row["home_team_id"])] = home_pts + hp
        points[(season, row["away_team_id"])] = away_pts + ap
    games["table_rank"] = rank_col
    games["opponent_table_rank"] = opp_rank_col
    return games

# Standings must be computed over ALL NWSL games that season (including away/knockout/COVID
# games for continuity), not just the filtered training set, then joined back in.
all_nwsl = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "games_with_transit.csv"), parse_dates=["date_time_utc"])
all_nwsl = all_nwsl[(all_nwsl["league"] == "nwsl") & all_nwsl["home_score"].notna() & all_nwsl["away_score"].notna()]
standings = running_standings(all_nwsl)[["game_id", "table_rank", "opponent_table_rank"]]
games = games.merge(standings, on="game_id", how="left")

=== 4. Running table_rank / opponent_table_rank (as-of-matchday, no leakage) ===


In [6]:
print("=== 5. days_since_last_home_game ===")
games = games.sort_values(["home_team_id", "date_time_utc"])
games["days_since_last_home_game"] = games.groupby("home_team_id")["date_time_utc"].diff().dt.days
games = games.sort_values("date_time_utc").reset_index(drop=True)

=== 5. days_since_last_home_game ===


In [7]:
print("=== 6. Join venue features, is_relocated_team flag for the CV split, finalize ===")
RELOCATED_TEAMS = {"Kansas City Current", "San Diego Wave FC", "Seattle Reign FC", "Washington Spirit"}
games = games.merge(
    venue_features[["stadium_id", "stadium_name", "capacity", "stadium_age_years",
                     "dedicated_soccer_facility", "shared_with_other_league",
                     "metro_size", "catchment_pop_30min", "catchment_pop_60min"]].rename(
        columns={"stadium_name": "venue_name", "capacity": "stadium_capacity"}
    ),
    on="stadium_id", how="left",
)
games["is_relocated_team"] = games["home_team"].isin(RELOCATED_TEAMS).astype(int)

# team_venue_tenure: seasons since THIS team moved into ITS CURRENT venue
team_seasons = games[["home_team", "year", "venue_name"]].drop_duplicates().sort_values(["home_team", "year"])
team_seasons["prev_venue"] = team_seasons.groupby("home_team")["venue_name"].shift(1)
team_seasons["is_move_season"] = team_seasons["prev_venue"].notna() & (team_seasons["venue_name"] != team_seasons["prev_venue"])

def _venue_tenure(g):
    move_years = g.loc[g["is_move_season"], "year"]
    if not len(move_years):
        return pd.Series(np.nan, index=g.index)
    start_year = move_years.max()  # most recent relocation
    return pd.Series(np.where(g["year"] >= start_year, g["year"] - start_year, np.nan), index=g.index)

team_seasons["team_venue_tenure"] = team_seasons.groupby("home_team", group_keys=False).apply(_venue_tenure)
games = games.merge(team_seasons[["home_team", "year", "team_venue_tenure"]], on=["home_team", "year"], how="left")

# team_league_tenure: seasons since THIS team's actual NWSL expansion, when known
PRE_2016_FOUNDING_TEAMS = {
    "Portland Thorns FC", "Seattle Reign FC", "Washington Spirit", "Chicago Stars FC",
    "NJ/NY Gotham FC", "Houston Dash",
}
expansion_year = teams.set_index("team_id")["expansion_year"]
games["_expansion_year"] = games["home_team_id"].map(expansion_year)
games.loc[games["home_team"].isin(PRE_2016_FOUNDING_TEAMS), "_expansion_year"] = np.nan
games["team_league_tenure"] = games["year"] - games["_expansion_year"]
games.loc[games["team_league_tenure"] < 0, "team_league_tenure"] = np.nan  # shouldn't happen, guards against bad joins
games = games.drop(columns=["_expansion_year"])

# venue_name is kept -> dropping team/venue identity was tested earlier and made
# LOTO generalization worse
FEATURE_COLS = [
    "table_rank", "opponent_table_rank", "days_since_last_home_game", "day_of_week", "year",
    "venue_name", "home_team", "away_team", "stadium_capacity",
    "metro_size", "dedicated_soccer_facility", "shared_with_other_league",
    "team_venue_tenure", "team_league_tenure",
]
out_cols = ["game_id", "date_time_utc", "season_name", "log_attendance", "attendance",
            "is_relocated_team"] + FEATURE_COLS + ["stadium_age_years",
                                                    "catchment_pop_30min", "catchment_pop_60min"]
final = games[out_cols].copy()

=== 6. Join venue features, is_relocated_team flag for the CV split, finalize ===


/var/folders/2y/rz_rt3v15kv2qz95_7stn2_r0000gn/T/ipykernel_32557/2037075300.py:32: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  team_seasons["team_venue_tenure"] = team_seasons.groupby("home_team", group_keys=False).apply(_venue_tenure)


In [8]:
print("=== 7. Flag and exclude capacity-censored rows (see 19_catboost_feature_engineering_v2.py) ===")
final["_season_key"] = final["home_team"] + "_" + final["year"].astype(str)
value_counts = final.groupby(["_season_key", "attendance"]).size().rename("n_repeats").reset_index()
repeated_values = value_counts[value_counts["n_repeats"] >= 3]
final = final.merge(repeated_values, on=["_season_key", "attendance"], how="left")
final["n_repeats"] = final["n_repeats"].fillna(0)
is_censored = (final["n_repeats"] >= 3) & (final["attendance"] >= 0.97 * final["stadium_capacity"])
print(f"  {int(is_censored.sum())} / {len(final)} rows flagged as capacity-censored, dropped")
final = final[~is_censored].drop(columns=["_season_key", "n_repeats"]).reset_index(drop=True)

out_path = os.path.join(DATA_PROCESSED_DIR, "catboost_training_table.csv")
final.to_csv(out_path, index=False)

print(f"\nWrote {out_path}: {len(final)} rows, {final['home_team'].nunique()} teams, "
      f"seasons {sorted(final['year'].unique())}")
print("\nMissingness by column:")
print(final[FEATURE_COLS + ["log_attendance"]].isna().mean().sort_values(ascending=False))
print("\nRows per home_team:")
print(final["home_team"].value_counts())

=== 7. Flag and exclude capacity-censored rows (see 19_catboost_feature_engineering_v2.py) ===
  50 / 1182 rows flagged as capacity-censored, dropped

Wrote ../data/processed/catboost_training_table.csv: 1132 rows, 14 teams, seasons [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]

Missingness by column:
team_venue_tenure            0.702297
team_league_tenure           0.562721
stadium_capacity             0.285336
metro_size                   0.160777
days_since_last_home_game    0.011484
table_rank                   0.000000
opponent_table_rank          0.000000
day_of_week                  0.000000
year                         0.000000
venue_name                   0.000000
home_team                    0.000000
away_team                    0.000000
dedicated_soccer_facility    0.000000
shared_with_other_league     0.000000
log_attendance               0.000000
dtype: float64

Rows per hom